# Lab 3 — Detección de anomalías con Machine Learning

**Autor:** David Robert Yucra Mamani  
**Curso:** Seguridad Informática  
**Modelo:** Isolation Forest  

El dataset se lee desde `lab3/network_traffic.csv`. La columna `label` se usa **solo para validación**, no para entrenamiento.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report
import joblib

BASE_DIR = Path.cwd()
if BASE_DIR.name != 'lab3':
    # Permite ejecutar desde la raiz del repositorio o desde lab3/
    DATA_PATH = BASE_DIR / 'lab3' / 'network_traffic.csv'
    MODEL_PATH = BASE_DIR / 'lab3' / 'modelo_anomalias.pkl'
else:
    DATA_PATH = BASE_DIR / 'network_traffic.csv'
    MODEL_PATH = BASE_DIR / 'modelo_anomalias.pkl'

print('Dataset:', DATA_PATH)

## 3.1 Exploración y preprocesamiento

In [ ]:
df = pd.read_csv(DATA_PATH)
print('Dimensiones:', df.shape)
display(df.head())
display(df.describe(include='all'))
print('
Valores nulos por columna:')
print(df.isna().sum())

In [ ]:
# Histogramas solicitados: bytes_sent y duration_sec
plt.figure(figsize=(9,5))
df['bytes_sent'].hist(bins=50)
plt.title('Distribución de bytes_sent')
plt.xlabel('bytes_sent')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()

plt.figure(figsize=(9,5))
df['duration_sec'].hist(bins=50)
plt.title('Distribución de duration_sec')
plt.xlabel('duration_sec')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()

In [ ]:
# Copia de trabajo y conversiones
work = df.copy()
work['timestamp'] = pd.to_datetime(work['timestamp'], errors='coerce')

# Columnas numéricas base
numeric_base = ['dst_port', 'bytes_sent', 'bytes_recv', 'duration_sec', 'packets']
for col in numeric_base:
    work[col] = pd.to_numeric(work[col], errors='coerce')
    work[col] = work[col].fillna(work[col].median())

# Tratamiento simple de atípicos extremos: clipping entre percentiles 1 y 99
clip_bounds = {}
for col in numeric_base:
    q01, q99 = work[col].quantile([0.01, 0.99])
    clip_bounds[col] = (float(q01), float(q99))
    work[col] = work[col].clip(q01, q99)

# Feature engineering solicitado: al menos 2 variables derivadas
work['hour'] = work['timestamp'].dt.hour.fillna(0)
work['dayofweek'] = work['timestamp'].dt.dayofweek.fillna(0)
work['ratio_bytes'] = work['bytes_sent'] / (work['bytes_recv'] + 1)
work['bytes_por_segundo'] = (work['bytes_sent'] + work['bytes_recv']) / (work['duration_sec'] + 1)
work['packets_por_segundo'] = work['packets'] / (work['duration_sec'] + 1)

# One-hot encoding del protocolo
protocol_dummies = pd.get_dummies(work['protocol'].fillna('UNKNOWN'), prefix='protocol')
features = pd.concat([
    work[['dst_port','bytes_sent','bytes_recv','duration_sec','packets','hour','dayofweek','ratio_bytes','bytes_por_segundo','packets_por_segundo']],
    protocol_dummies
], axis=1)

features = features.replace([np.inf, -np.inf], np.nan).fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

print('Features usadas para entrenamiento:', list(features.columns))
print('Matriz normalizada:', X_scaled.shape)

## 3.2 Entrenamiento del modelo y métricas

In [ ]:
# Entrenamiento sin usar label
model = IsolationForest(contamination=0.05, n_estimators=100, random_state=42)
model.fit(X_scaled)

pred = model.predict(X_scaled)  # -1 = anomalía, 1 = normal

# Validación con label solo después del entrenamiento
label_map = {'anomaly': -1, 'normal': 1, 'anomalia': -1, 'normal': 1}
y_true = work['label'].astype(str).str.lower().map(label_map)
if y_true.isna().any():
    print('Advertencia: existen labels no reconocidos. Valores:', work['label'].unique())
    y_true = y_true.fillna(1)
y_true = y_true.astype(int)

precision = precision_score(y_true, pred, pos_label=-1, zero_division=0)
recall = recall_score(y_true, pred, pos_label=-1, zero_division=0)
f1 = f1_score(y_true, pred, pos_label=-1, zero_division=0)

print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1-Score:  {f1:.4f}')
print('
Reporte de clasificación:')
print(classification_report(y_true, pred, labels=[-1,1], target_names=['anomaly','normal'], zero_division=0))

cm = confusion_matrix(y_true, pred, labels=[-1, 1])
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=['Pred anomaly','Pred normal'], yticklabels=['Real anomaly','Real normal'])
plt.title('Matriz de confusión - Isolation Forest')
plt.tight_layout()
plt.show()

## 3.3 Interpretación y umbral dinámico

En Isolation Forest, los valores más bajos del `decision_function` suelen indicar mayor anomalía.

In [ ]:
scores = model.decision_function(X_scaled)
work['score_anomalia'] = scores
work['prediccion_base'] = pred

plt.figure(figsize=(11,5))
plt.plot(scores, marker='.', linewidth=0)
plt.title('Score de anomalía por registro')
plt.xlabel('Registro')
plt.ylabel('decision_function')
plt.tight_layout()
plt.show()

In [ ]:
thresholds = np.linspace(scores.min(), scores.max(), 120)
f1_values = []
for th in thresholds:
    pred_th = np.where(scores < th, -1, 1)
    f1_values.append(f1_score(y_true, pred_th, pos_label=-1, zero_division=0))

best_idx = int(np.argmax(f1_values))
best_threshold = float(thresholds[best_idx])
best_f1 = float(f1_values[best_idx])

plt.figure(figsize=(10,5))
plt.plot(thresholds, f1_values)
plt.axvline(best_threshold, linestyle='--')
plt.title(f'Curva umbral vs F1 — mejor umbral={best_threshold:.5f}, F1={best_f1:.4f}')
plt.xlabel('Umbral sobre decision_function')
plt.ylabel('F1-Score')
plt.tight_layout()
plt.show()

print('Mejor umbral:', best_threshold)
print('Mejor F1:', best_f1)

In [ ]:
# Top 10 registros más anómalos: menor score
cols_mostrar = list(df.columns) + ['score_anomalia', 'prediccion_base']
top10_anomalias = work.sort_values('score_anomalia').head(10)[cols_mostrar]
display(top10_anomalias)

### Interpretación de posibles amenazas

Los registros del Top 10 pueden representar una amenaza real cuando combinan uno o varios de estos factores:

- `bytes_sent` demasiado alto frente a `bytes_recv`, lo que puede sugerir exfiltración de datos.
- `duration_sec` muy corto con muchos paquetes, posible automatización o escaneo.
- Puertos de destino poco comunes o sensibles.
- Comunicación fuera del patrón normal del tráfico de la red.
- Alta tasa `bytes_por_segundo` o `packets_por_segundo`, que puede indicar transferencia masiva o comportamiento anómalo.

## 3.4 Exportación del modelo

In [ ]:
paquete_modelo = {
    'model': model,
    'scaler': scaler,
    'feature_columns': list(features.columns),
    'threshold': best_threshold,
    'clip_bounds': clip_bounds,
    'nota': 'Modelo Isolation Forest para detección de anomalías. -1=anomalia, 1=normal.'
}
joblib.dump(paquete_modelo, MODEL_PATH)
print('Modelo guardado en:', MODEL_PATH)

## Prueba del script `predecir.py`

Desde terminal:

```bash
cd examen-practico-yucra
source .venv/bin/activate
python lab3/predecir.py lab3/nuevo_trafico.csv
```